In [ ]:
"""
Parte4_shap.py -- 4.2 e 4.3: o que o modelo aprendeu e se bate com a teoria.

Para cada modelo (alvo, mes, bloco): TreeSHAP nativo do LightGBM (pred_contrib=True) APENAS nas obs de teste
daquele modelo. Propriedade conferida em cada modelo:  soma das contribuicoes + base = pred_log da Parte 2.
Normalizacao por observacao:  imp_ij = |phi_ij| / sum_k |phi_ik|   (soma 100% por obs).
Agrega por variavel (media), por categoria do dicionario da Parte 1 (soma das variaveis) e por periodo.
Sinal: Spearman (feature x alvo) por mes, medio; e Spearman (feature x seu SHAP), que da a direcao no modelo.
Saidas em resultados_parte4/: shap_por_variavel.csv, shap_por_categoria.csv, shap_por_periodo.csv,
quadro_teoria_modelo.csv
"""
import unicodedata
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import comum_parte4 as C

TOL_SOMA = 1e-6
MIN_OBS_SPEARMAN = 10

# sinais esperados (roteiro 4.3, modelo de Gordon). "+", "-", "ambiguo" ou "?" (nao especificado no roteiro)
SINAL_CATEGORIA = {"lucratividade": "+", "crescimento": "+", "solidez financeira": "ambiguo", "tamanho": "-"}
GIRO = ("giro_ativos", "at_turn_wr", "assetturnover_an")
BETA = ("beta", "CAPMbeta")
SINAL_VARIAVEL = {**{(g, "lnv2a"): "+" for g in GIRO}, **{(g, "lnv2s"): "-" for g in GIRO},
                  **{(g, "lnm2b"): "?" for g in GIRO},
                  **{(b, a): "-" for b in BETA for a in C.ALVOS}}      # beta: custo de capital (Gordon)


def _norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()


def shap_modelo(painel_idx, prev_modelo, alvo, data, bloco):
    booster = C.carrega_modelo(alvo, data, bloco)
    X = C.matriz_features(painel_idx.loc[data].loc[prev_modelo["ticker"]], booster)
    contrib = booster.predict(X, pred_contrib=True)
    phi, base = contrib[:, :-1], contrib[:, -1]
    desvio = np.abs(phi.sum(1) + base - prev_modelo["pred_log"].to_numpy()).max()
    if desvio > TOL_SOMA:
        raise RuntimeError(f"SHAP nao reproduz pred_log da Parte 2 em {alvo} {data:%Y-%m} b{bloco} "
                           f"(desvio {desvio:.2e}): modelo/painel diferentes dos usados no treino?")
    ids = pd.DataFrame({"ticker": prev_modelo["ticker"].to_numpy(), "data": data, "alvo": alvo,
                        "log_real": prev_modelo["log_real"].to_numpy()})
    return pd.concat([ids, pd.DataFrame(phi, columns=booster.feature_name())], axis=1), X.reset_index(drop=True)


def calcula_shap(painel, prev):
    """Retorna phi (contribuicoes) e X (valores das features), alinhados linha a linha."""
    painel_idx = painel.set_index(["data", "ticker"]).sort_index()
    phis, xs = [], []
    for (alvo, data, bloco), g in prev.groupby(["alvo", "data", "bloco"]):
        phi, X = shap_modelo(painel_idx, g, alvo, data, bloco)
        phis.append(phi)
        xs.append(X)
    return pd.concat(phis, ignore_index=True), pd.concat(xs, ignore_index=True)


def normaliza(phi, feats):
    a = phi[feats].abs()
    tot = a.sum(axis=1).replace(0, np.nan)
    return pd.concat([phi[["ticker", "data", "alvo"]], a.div(tot, axis=0)], axis=1)


def categorias(feats, dic):
    mapa = dict(zip(dic["nome"], dic["categoria"]))
    sem = [f for f in feats if f not in mapa]
    if sem:
        warnings.warn(f"{len(sem)} variaveis sem categoria no dicionario -> 'Sem categoria': {sem[:10]}")
    return {f: mapa.get(f, "Sem categoria") for f in feats}


def periodo(ano):
    for nome, (a, b) in C.PERIODOS.items():
        if a <= ano <= b:
            return nome
    return "outro"


def sinais(phi, X, feats):
    """Spearman medio por mes: feature x alvo (pedido no roteiro) e feature x SHAP da feature (direcao no modelo).
    phi e X estao alinhados linha a linha (mesma ordem de calcula_shap)."""
    warnings.filterwarnings("ignore", message="An input array is constant")   # SHAP nulo no mes: rho = NaN
    linhas = []
    grupos = phi.groupby(["alvo", "data"]).indices
    for f in feats:
        if not pd.api.types.is_numeric_dtype(X[f]):
            continue
        x, y, s = X[f].to_numpy(float), phi["log_real"].to_numpy(float), phi[f].to_numpy(float)
        for (alvo, _), idx in grupos.items():
            ok = np.isfinite(x[idx]) & np.isfinite(y[idx])
            if ok.sum() < MIN_OBS_SPEARMAN or np.unique(x[idx][ok]).size < 3:
                continue
            linhas.append((f, alvo, spearmanr(x[idx][ok], y[idx][ok]).statistic,
                           spearmanr(x[idx][ok], s[idx][ok]).statistic))
    s = pd.DataFrame(linhas, columns=["variavel", "alvo", "rho_alvo", "rho_shap"])
    return s.groupby(["variavel", "alvo"]).agg(spearman_alvo=("rho_alvo", "mean"),
                                               frac_meses_pos=("rho_alvo", lambda v: (v > 0).mean()),
                                               spearman_shap=("rho_shap", "mean")).reset_index()


def sinal_txt(x, tol=0.05):
    return "+" if x > tol else "-" if x < -tol else "~0"


def quadro_teoria(sig, imp_var, cat):
    q = sig.merge(imp_var, on=["variavel", "alvo"], how="left")
    q["categoria"] = q["variavel"].map(cat)
    q["sinal_esperado"] = [SINAL_VARIAVEL.get((v, a), SINAL_CATEGORIA.get(_norm(c), "?"))
                           for v, a, c in zip(q["variavel"], q["alvo"], q["categoria"])]
    q["sinal_alvo"] = q["spearman_alvo"].map(sinal_txt)
    q["sinal_modelo"] = q["spearman_shap"].map(sinal_txt)
    q["confere"] = np.where(q["sinal_esperado"].isin(["+", "-"]), q["sinal_esperado"] == q["sinal_modelo"], np.nan)
    cols = ["alvo", "categoria", "variavel", "importancia_pct", "sinal_esperado", "spearman_alvo", "frac_meses_pos",
            "spearman_shap", "sinal_alvo", "sinal_modelo", "confere"]
    return q[cols].sort_values(["alvo", "importancia_pct"], ascending=[True, False])


def main():
    C.PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    painel, prev, dic = C.ler_painel(), C.ler_previsoes(), C.ler_dicionario()
    phi, X = calcula_shap(painel, prev)
    feats = [c for c in phi.columns if c not in ("ticker", "data", "alvo", "log_real")]
    print(f"SHAP: {len(phi)} obs de teste | {len(feats)} variaveis | soma = previsao conferida em todos os modelos")
    imp = normaliza(phi, feats)
    cat = categorias(feats, dic)

    por_var = imp.groupby("alvo")[feats].mean().T.mul(100)
    imp_var = por_var.reset_index(names="variavel").melt(id_vars="variavel", var_name="alvo", value_name="importancia_pct")
    por_cat = por_var.groupby(cat).sum()
    imp["periodo"] = imp["data"].dt.year.map(periodo)
    por_per = imp.groupby(["alvo", "periodo"])[feats].mean().T.mul(100).groupby(cat).sum()
    por_per = (por_per.T.reset_index().melt(id_vars=["alvo", "periodo"], var_name="categoria", value_name="importancia_pct")
               .pivot_table(index=["alvo", "categoria"], columns="periodo", values="importancia_pct"))
    quadro = quadro_teoria(sinais(phi, X, feats), imp_var, cat)

    ordem = C.ALVOS[0] if C.ALVOS[0] in por_var else por_var.columns[0]
    por_var.round(2).sort_values(ordem, ascending=False).to_csv(C.PASTA_SAIDA / "shap_por_variavel.csv")
    por_cat.round(2).sort_values(ordem, ascending=False).to_csv(C.PASTA_SAIDA / "shap_por_categoria.csv")
    por_per.round(2).to_csv(C.PASTA_SAIDA / "shap_por_periodo.csv")
    quadro.round(3).to_csv(C.PASTA_SAIDA / "quadro_teoria_modelo.csv", index=False)

    pd.set_option("display.width", 220, "display.max_columns", 30)
    print("\nImportancia por categoria (% medio do |SHAP| normalizado):")
    print(por_cat.round(1).sort_values(ordem, ascending=False).to_string())
    print("\nTop 5 variaveis por alvo:")
    for a in por_var.columns:
        print(f"  {a}:", ", ".join(f"{v} {x:.1f}%" for v, x in por_var[a].nlargest(5).items()))
    defin = quadro[quadro["sinal_esperado"].isin(["+", "-"])]
    print(f"\nQuadro teoria x modelo: {int(defin['confere'].sum())} de {len(defin)} sinais definidos conferem "
          f"(detalhe em quadro_teoria_modelo.csv)")


if __name__ == "__main__":
    main()